In [ ]:
import json
import numpy as np
import pandas as pd
import time
np.random.seed()

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

def MC_sample(t, current_state, transition_dict):
    """ Draw one next state given current time t and current_state """

    # Convert keys to strings
    t_key     = str(t)
    state_key = str(current_state)
    # Get all possible transitions
    probs_dict = transition_dict[t_key][state_key]
    # Extract valid next states and probabilities
    states_str = [s for s, p in probs_dict.items() if p > 0]
    weights    = [p for s, p in probs_dict.items() if p > 0]
    states = [eval(s) for s in states_str] # Convert state strings to tuples
    idx    = np.random.choice(len(states), p=weights) # Randomly sample next state
    return states[idx]

def create_OSpaths(nbOS, T_Bef, k_init, transition_dict, output_file):
    """
    Generate Out-of-Sample (OOS) paths using transition probabilities.
    Saves a CSV file where each cell contains a stringified tuple representing a state.
    """
    T = len(T_Bef)
    OS_paths = np.full((nbOS, T), None, dtype=object)

    # Initialize first state for all paths
    OS_paths[:, 0] = k_init

    # Generate paths using MC sampling
    for s in range(nbOS):
        for t in range(1, T):
            prev_state = OS_paths[s, t-1]
            next_state = MC_sample(t-1, prev_state, transition_dict)
            OS_paths[s, t] = next_state

    # Save the paths as CSV
    df = pd.DataFrame(OS_paths)
    df.to_csv(output_file, index=False)

    print(f"[INFO] OOS paths generated and saved to: {output_file}")
    return OS_paths

In [ ]:
import numpy as np
xval = [[0] * 5 for _ in range(3)]
xval1 = np.zeros((3, 5))
print(xval)
print(xval1)

[[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [ ]:
import pandas as pd
sample_paths_file = "Data/real_sample_paths.csv"
OS_paths_df = pd.read_csv(sample_paths_file)
OS_paths = OS_paths_df.values[:5]
print(OS_paths)
k_t = eval(OS_paths[0, 10])
print(k_t)

[['(0.0, 0)' '(-23.55, 0)' '(-32.52, 0)' '(-24.2, 0)' '(-30.36, 0)'
  '(-19.43, 2)' '(-28.83, 3)' '(-51.09, 3)' '(-83.68, 3)' '(-160.18, 3)'
  '(-231.77, 3)']
 ['(0.0, 0)' '(-23.55, 1)' '(-32.52, 1)' '(-24.2, 1)' '(-30.36, 0)'
  '(-19.43, 1)' '(-28.83, 1)' '(-16.36, 1)' '(-41.1, 2)' '(-26.59, 2)'
  '(-33.68, 2)']
 ['(0.0, 0)' '(-23.55, 1)' '(10.24, 3)' '(21.96, 2)' '(49.02, 1)'
  '(24.49, 1)' '(11.5, 0)' '(24.12, 1)' '(14.49, 1)' '(12.43, 1)'
  '(18.16, 2)']
 ['(0.0, 0)' '(23.55, 1)' '(34.28, 1)' '(21.96, 0)' '(19.13, 1)'
  '(24.49, 2)' '(11.5, 0)' '(24.12, 0)' '(14.49, 1)' '(12.43, 1)'
  '(5.28, 1)']
 ['(0.0, 0)' '(23.55, 0)' '(10.24, 0)' '(21.96, 0)' '(19.13, 1)'
  '(24.49, 2)' '(11.5, 3)' '(24.12, 4)' '(14.49, 5)' '(12.43, 4)'
  '(18.16, 4)']]
(-33.68, 2)


In [13]:
import numpy as np
np.random.seed()
print(np.random.randint(0, 100, 5))

[42 64 12 36 57]


In [9]:
T = 10
for t in range(9, 0, -1):
    print(t)

9
8
7
6
5
4
3
2
1


In [ ]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    transition_dict = json.load(f)

nodeLists = {int(t): [eval(s) for s in trans.keys()] 
             for t, trans in transition_dict.items()}

# All Markov states reachable at stage t
print(nodeLists[2])

[(-32.52, 0), (-32.52, 1), (-32.52, 2), (-32.52, 3), (-32.52, 4), (-32.52, 5), (10.24, 0), (10.24, 1), (10.24, 2), (10.24, 3), (10.24, 4), (10.24, 5), (34.28, 0), (34.28, 1), (34.28, 2), (34.28, 3), (34.28, 4), (34.28, 5)]


In [12]:
import json

with open("Data/pi_mssp_d_ST_track_15.json", "r") as f:
    data = json.load(f)

print("Top-level keys (stages):", list(data.keys())[:10])
first_stage = list(data.keys())[0]
print("Sample stage:", first_stage)
print("Sample states at stage 0:", list(data[first_stage].keys())[:5])

first_state = list(data[first_stage].keys())[0]
print("Sample transitions from", first_state, ":", data[first_stage][first_state])


Top-level keys (stages): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Sample stage: 0
Sample states at stage 0: ['(0.0, 0)']
Sample transitions from (0.0, 0) : {'(-23.55, 0)': 0.256, '(-23.55, 1)': 0.203, '(-23.55, 2)': 0.023, '(-23.55, 3)': 0.0, '(-23.55, 4)': 0.0, '(-23.55, 5)': 0.0, '(23.55, 0)': 0.275, '(23.55, 1)': 0.219, '(23.55, 2)': 0.024, '(23.55, 3)': 0.0, '(23.55, 4)': 0.0, '(23.55, 5)': 0.0}


In [14]:
import json

def load_transition_dict(path):
    with open(path, "r") as f:
        return json.load(f)

def build_node_lists(transition_dict):
    nodeLists = {}
    for t in transition_dict:
        nodeLists[int(t)] = [eval(s) for s in transition_dict[t].keys()]
    return nodeLists

# Load transition dictionary and build nodeLists
transition_dict = load_transition_dict("Data/pi_mssp_d_ST_track_15.json")
nodeLists = build_node_lists(transition_dict)

state_to_idx = {}
idx = 0
for t in nodeLists:
    for k in nodeLists[t]:
        state_to_idx[k] = idx
        idx += 1

K = len(state_to_idx)
print(K)

t = 8
n = list(transition_dict[str(t)].keys())[0]
print("Current state at t =", n)
print("Reachable next states at t+1:")
print(transition_dict[str(t)][n])



343
Current state at t = (-526.81, 0)
Reachable next states at t+1:
{'(-1110.62, 0)': 0.249, '(-1110.62, 1)': 0.198, '(-1110.62, 2)': 0.022, '(-1110.62, 3)': 0.0, '(-1110.62, 4)': 0.0, '(-1110.62, 5)': 0.0, '(-360.84, 0)': 0.266, '(-360.84, 1)': 0.211, '(-360.84, 2)': 0.024, '(-360.84, 3)': 0.0, '(-360.84, 4)': 0.0, '(-360.84, 5)': 0.0, '(-160.18, 0)': 0.016, '(-160.18, 1)': 0.013, '(-160.18, 2)': 0.001, '(-160.18, 3)': 0.0, '(-160.18, 4)': 0.0, '(-160.18, 5)': 0.0, '(-69.33, 0)': 0.0, '(-69.33, 1)': 0.0, '(-69.33, 2)': 0.0, '(-69.33, 3)': 0.0, '(-69.33, 4)': 0.0, '(-69.33, 5)': 0.0, '(-26.59, 0)': 0.0, '(-26.59, 1)': 0.0, '(-26.59, 2)': 0.0, '(-26.59, 3)': 0.0, '(-26.59, 4)': 0.0, '(-26.59, 5)': 0.0, '(-8.75, 0)': 0.0, '(-8.75, 1)': 0.0, '(-8.75, 2)': 0.0, '(-8.75, 3)': 0.0, '(-8.75, 4)': 0.0, '(-8.75, 5)': 0.0, '(12.43, 0)': 0.0, '(12.43, 1)': 0.0, '(12.43, 2)': 0.0, '(12.43, 3)': 0.0, '(12.43, 4)': 0.0, '(12.43, 5)': 0.0, '(35.9, 0)': 0.0, '(35.9, 1)': 0.0, '(35.9, 2)': 0.0, '(35.9,

In [22]:
import numpy as np

def MC_sample_json(t, current_state, transition_dict):
    """
    Return ONE next state given (t , current_state).

    Parameters
    ----------
    t : int
        Current time step (0-based, as in your forward pass).
    current_state : tuple
        Current Markov state, e.g. (23.55, 1).
    transition_dict : dict
        Loaded JSON dictionary of transition probabilities.

    Returns
    -------
    tuple
        Next state sampled according to transition probabilities.
    """
    # Keys in the JSON are strings
    t_key       = str(t)
    state_key   = str(current_state)

    probs_dict  = transition_dict[t_key][state_key]          # { "(next_state)": prob, ... }
    states_str  = [s for s, p in probs_dict.items() if p > 0]
    weights     = [p for s, p in probs_dict.items() if p > 0]

    states      = [eval(s) for s in states_str]              # back to tuple form
    idx         = np.random.choice(len(states), p=weights)   # Monte-Carlo draw
    return states[idx]


In [33]:
# --- 2. Set known values -------------------------------------------------
t              = 1              # current time step
current_state  = (-23.55, 0)       # deterministic starting state at t = 0

# --- 3. Call the single-step sampler ------------------------------------
next_state = MC_sample_json(t, current_state, transition_dict)

print(f"Current state (t={t}): {current_state}")
print(f"Sampled next state   : {next_state}")

Current state (t=1): (-23.55, 0)
Sampled next state   : (10.24, 0)
